In [1]:
import os

import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.datasets import VisionDataset
from torchvision.transforms import v2



In [2]:
torch.manual_seed(3407)
torch.cuda.manual_seed(3407)

cudnn.deterministic = True
cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

base_dir = "/kaggle/input/cassava-leaf-disease-classification"
train_csv_path = f"{base_dir}/train.csv"
train_dir = f"{base_dir}/train_images"
test_dir = f"{base_dir}/test_images"
sample_sub_path = f"{base_dir}/sample_submission.csv"

img_size = 224
batch_size = 32
num_workers = 4
num_classes = 5

epochs = 4
lr = 3e-4

label_smoothing = 0.05
grad_clip_norm = 1.0


def resolve_dir(primary: str, fallback: str) -> str:
    # Bugfix: some dataset copies have an extra nested folder (e.g., test_images/test_images).
    # Keep paths unchanged unless the primary doesn't exist.
    if os.path.isdir(primary):
        return primary
    if os.path.isdir(fallback):
        return fallback
    raise FileNotFoundError(
        f"Could not find directory. Tried: {primary} and {fallback}"
    )


train_dir = resolve_dir(train_dir, os.path.join(train_dir, "train_images"))
test_dir = resolve_dir(test_dir, os.path.join(test_dir, "test_images"))

print("Resolved train_dir:", train_dir)
print("Resolved test_dir :", test_dir)



cuda
Resolved train_dir: /kaggle/input/cassava-leaf-disease-classification/train_images
Resolved test_dir : /kaggle/input/cassava-leaf-disease-classification/test_images


In [3]:
from torchvision.models import vit_b_16, ViT_B_16_Weights

weights = ViT_B_16_Weights.DEFAULT
vit_model = vit_b_16(weights=weights)
vit_model.heads.head = torch.nn.Linear(vit_model.heads.head.in_features, num_classes)
vit_model.to(device)




Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:02<00:00, 116MB/s]


VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [4]:
class CassavaTrainDataset(VisionDataset):
    """Train dataset reading labels from train.csv and images from train_images/."""

    def __init__(self, data_dir, df, transform=None):
        super().__init__(root=data_dir)
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row["image_id"]
        label = int(row["label"])
        img = Image.open(os.path.join(self.root, filename)).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.df)


class CassavaTestDataset(VisionDataset):
    """Test dataset with deterministic TTA list; uses sample_submission order.

    Returns:
      imgs: Tensor [n_tta, C, H, W]
      filename: str
    """

    def __init__(self, data_dir, image_ids, transform=None, ttas=None):
        super().__init__(root=data_dir)
        self.transform = transform
        self.images = list(image_ids)
        self.ttas = ttas if ttas is not None else [lambda x: x]

    def __getitem__(self, idx):
        filename = self.images[idx]
        img_path = os.path.join(self.root, filename)
        img = Image.open(img_path).convert("RGB")
        imgs = [self.transform(t(img)) for t in self.ttas]
        imgs = torch.stack(imgs, dim=0)
        return imgs, filename

    def __len__(self):
        return len(self.images)




In [5]:
# Bugfix: weights.meta may not expose 'mean'/'std' keys in this torchvision build.
# Use the official transforms() for eval preprocessing; keep core logic the same.
base_eval_tfms = weights.transforms()

# Keep the same augmentation intent but avoid relying on weights.meta for normalization.
# (Normalization matches ViT weights via base_eval_tfms in val/test; train uses same mean/std constants.)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transforms = v2.Compose(
    [
        v2.RandomResizedCrop(
            size=(img_size, img_size), scale=(0.75, 1.0), ratio=(0.9, 1.1)
        ),
        v2.RandomHorizontalFlip(p=0.5),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
test_transforms = base_eval_tfms


def tta_identity(img):
    return img


def tta_hflip(img):
    return img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)


def tta_vflip(img):
    return img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)


def tta_rot90(img):
    return img.transpose(Image.Transpose.ROTATE_90)


def tta_rot180(img):
    return img.transpose(Image.Transpose.ROTATE_180)


def tta_rot270(img):
    return img.transpose(Image.Transpose.ROTATE_270)


ttas = [tta_identity, tta_hflip, tta_vflip, tta_rot90, tta_rot180, tta_rot270]

train_df = pd.read_csv(train_csv_path)


def stratified_split(df, label_col="label", val_frac=0.1, seed=3407):
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_parts = []
    tr_parts = []
    for _, g in df.groupby(label_col):
        n_val = max(1, int(round(len(g) * val_frac)))
        val_parts.append(g.iloc[:n_val])
        tr_parts.append(g.iloc[n_val:])
    val_df = (
        pd.concat(val_parts, axis=0)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )
    tr_df = (
        pd.concat(tr_parts, axis=0)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )
    return tr_df, val_df


tr_df, val_df = stratified_split(train_df, val_frac=0.1, seed=3407)

train_dataset = CassavaTrainDataset(train_dir, tr_df, transform=train_transforms)
val_dataset = CassavaTrainDataset(train_dir, val_df, transform=test_transforms)

class_counts = tr_df["label"].value_counts().sort_index()
class_weights = (1.0 / class_counts).to_dict()
sample_weights = tr_df["label"].map(class_weights).astype("float32").values
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,  # sampler controls order
    sampler=sampler,
    num_workers=num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

sample_sub = pd.read_csv(sample_sub_path)
test_dataset = CassavaTestDataset(
    test_dir,
    image_ids=sample_sub["image_id"].tolist(),
    transform=test_transforms,
    ttas=ttas,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

normalizer = torch.nn.Softmax(dim=1)

print("Train/Val/Test sizes:", len(train_dataset), len(val_dataset), len(test_dataset))



Train/Val/Test sizes: 16849 1872 2676


In [6]:
import torch.nn.functional as F


def set_backbone_trainable(model, trainable: bool):
    for name, p in model.named_parameters():
        if name.startswith("heads."):
            p.requires_grad = True
        else:
            p.requires_grad = trainable


set_backbone_trainable(vit_model, trainable=False)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, vit_model.parameters()), lr=lr, weight_decay=0.05
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs * max(1, len(train_loader))
)

for epoch in range(epochs):
    if epoch == 1:
        set_backbone_trainable(vit_model, trainable=True)
        optimizer = torch.optim.AdamW(vit_model.parameters(), lr=lr, weight_decay=0.05)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=(epochs - epoch) * max(1, len(train_loader))
        )

    vit_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = vit_model(xb)

        loss = F.cross_entropy(logits, yb, label_smoothing=label_smoothing)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(vit_model.parameters(), max_norm=grad_clip_norm)

        optimizer.step()
        scheduler.step()

        running_loss += float(loss.item()) * xb.size(0)
        preds = logits.argmax(dim=1)
        correct += int((preds == yb).sum().item())
        total += int(xb.size(0))

    vit_model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = vit_model(xb)
            preds = logits.argmax(dim=1)
            val_correct += int((preds == yb).sum().item())
            val_total += int(xb.size(0))

    print(
        f"epoch {epoch+1}/{epochs} | "
        f"loss {running_loss/total:.4f} | "
        f"train_acc {correct/total:.4f} | "
        f"val_acc {val_correct/max(1,val_total):.4f}"
    )



epoch 1/4 | loss 1.1860 | train_acc 0.5547 | val_acc 0.6928
epoch 2/4 | loss 1.2329 | train_acc 0.5191 | val_acc 0.6207
epoch 3/4 | loss 0.9201 | train_acc 0.6863 | val_acc 0.7126
epoch 4/4 | loss 0.7223 | train_acc 0.7844 | val_acc 0.7853


In [7]:
all_names = []
all_preds = []

vit_model.eval()

with torch.no_grad():
    for batch_idx, (inputs, filenames) in enumerate(test_loader):
        bsz, n_tta, c, h, w = inputs.shape
        x = inputs.view(bsz * n_tta, c, h, w).to(device, non_blocking=True)

        probs = normalizer(vit_model(x))  # [B*n_tta, num_classes]
        probs = probs.view(bsz, n_tta, -1).mean(dim=1)  # [B, num_classes]

        pred_labels = torch.argmax(probs, dim=1).tolist()

        all_names.extend(list(filenames))
        all_preds.extend(pred_labels)

print("Predicted test images:", len(all_names))



Predicted test images: 2676


In [8]:
pred_map = dict(zip(all_names, all_preds))

missing = [
    img_id for img_id in sample_sub["image_id"].tolist() if img_id not in pred_map
]
if len(missing) > 0:
    # Extra robustness: if images are missing on disk, fail fast with a clear message.
    existing = set(os.listdir(test_dir))
    missing_on_disk = [m for m in missing if m not in existing]
    raise RuntimeError(
        f"Missing predictions for {len(missing)} test images. "
        f"Missing on disk (first 5): {missing_on_disk[:5]}"
    )

my_submission = pd.DataFrame(
    {
        "image_id": sample_sub["image_id"],
        "label": sample_sub["image_id"].map(pred_map).astype(int),
    }
)
my_submission.to_csv("submission.csv", index=False)

print(my_submission.head())
print("Wrote submission.csv with rows:", len(my_submission))

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
Wrote submission.csv with rows: 2676
